### import essential libraries

In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from imblearn.over_sampling import RandomOverSampler
from sklearn.utils import resample
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

### Load Data

In [2]:
iris = load_iris()

In [3]:
X, y = iris['data'], iris['target']

### turning the problem to binary classsification

In [4]:
# Combine class 0 and 1 as the first class, class 2 as the second class
y = np.where(np.isin(y, [0, 1]), 0, 1)

### Preprocessing

In [5]:
# Standardize the data
scaler = StandardScaler()
X = scaler.fit_transform(X)

### train- test split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Define imbalance ratios

In [7]:
imbalance_ratios = [(1, 99), (5, 95), (10, 90), (20, 80), (30, 70)]

### Define a function to imbalance the data

In [8]:
def create_imbalanced_dataset(X, y, ratio):
    unique_classes = np.unique(y)
    if len(unique_classes) < 2:
        raise ValueError('The dataset must contain at least two classes.')
    
    X_imbalanced, y_imbalanced = [], []
    for i, cls in enumerate(unique_classes):
        samples = X[y == cls]
        n_samples = min(max(int(len(X) * ratio[i] / (ratio[0] + ratio[1])), 1), len(samples))
        
        resampled_samples = resample(samples, replace=False, n_samples=n_samples, random_state=42)
        
        X_imbalanced.append(resampled_samples)
        y_imbalanced.extend([cls] * n_samples)
    
    X_imbalanced = np.vstack(X_imbalanced)
    y_imbalanced = np.array(y_imbalanced)
    
    return X_imbalanced, y_imbalanced

### for each ratio implement three ideas and evaluate them

In [9]:
for ratio in imbalance_ratios:
    
    # Create imbalanced training set
    X_train_imbalanced, y_train_imbalanced = create_imbalanced_dataset(X_train, y_train, ratio)


    # Train and evaluate logistic regression on imbalanced dataset
    clf_imbalanced = LogisticRegression(max_iter=1000, random_state=42)
    clf_imbalanced.fit(X_train_imbalanced, y_train_imbalanced)
    y_pred_imbalanced = clf_imbalanced.predict(X_test)
    

    # Apply RandomOverSampler to balance the dataset by oversampling the minority class
    random_over_sampler = RandomOverSampler(sampling_strategy='minority', random_state=42)
    X_train_balanced_with_ros, y_train_balanced_with_ros = random_over_sampler.fit_resample(X_train_imbalanced, y_train_imbalanced)

    # Train and evaluate logistic regression on balanced dataset (with randomm over sampling)
    model = LogisticRegression(max_iter=1000,random_state=42)
    model.fit(X_train_balanced_with_ros, y_train_balanced_with_ros)
    y_pred_balanced_with_ros = model.predict(X_test)


    # Create and train the logistic regression model with class weighting
    model_with_class_weighting = LogisticRegression(class_weight='balanced', random_state=42)
    model_with_class_weighting.fit(X_train_imbalanced, y_train_imbalanced)
    y_pred_balanced_with_class_weighting = model_with_class_weighting.predict(X_test)

    

    # Train and evaluate XGBClassifier with logistic regression as the base learner
    xgb_clf = XGBClassifier(booster='gblinear', objective='binary:logistic', n_estimators=100, learning_rate=0.1, random_state=42)
    xgb_clf.fit(X_train_imbalanced, y_train_imbalanced)
    y_pred_balanced_with_xgb = xgb_clf.predict(X_test)


    # Calculate performance metrics
    metrics_imbalanced = [accuracy_score(y_test, y_pred_imbalanced), precision_score(y_test, y_pred_imbalanced), recall_score(y_test, y_pred_imbalanced), f1_score(y_test, y_pred_imbalanced)]
    metrics_balanced_with_ros = [accuracy_score(y_test, y_pred_balanced_with_ros), precision_score(y_test, y_pred_balanced_with_ros), recall_score(y_test, y_pred_balanced_with_ros), f1_score(y_test, y_pred_balanced_with_ros)]
    metrics_balanced_with_class_weighting = [accuracy_score(y_test, y_pred_balanced_with_class_weighting), precision_score(y_test, y_pred_balanced_with_class_weighting), recall_score(y_test, y_pred_balanced_with_class_weighting), f1_score(y_test, y_pred_balanced_with_class_weighting)]
    metrics_balanced_with_xgb= [accuracy_score(y_test, y_pred_balanced_with_xgb), precision_score(y_test, y_pred_balanced_with_xgb), recall_score(y_test, y_pred_balanced_with_xgb), f1_score(y_test, y_pred_balanced_with_xgb)]

    print(f"Imbalance ratio: {ratio[0]}:{ratio[1]}")
    print("Imbalanced dataset metrics: Accuracy: {:.4f}, Precision: {:.4f}, Recall: {:.4f}, F1-score: {:.4f}".format(*metrics_imbalanced))
    print("Balanced dataset with random over sampling metrics: Accuracy: {:.4f}, Precision: {:.4f}, Recall: {:.4f}, F1-score: {:.4f}".format(*metrics_balanced_with_ros))
    print("Balanced dataset with class weighting metrics: Accuracy: {:.4f}, Precision: {:.4f}, Recall: {:.4f}, F1-score: {:.4f}".format(*metrics_balanced_with_class_weighting))
    print("Balanced dataset with xgb metrics: Accuracy: {:.4f}, Precision: {:.4f}, Recall: {:.4f}, F1-score: {:.4f}".format(*metrics_balanced_with_xgb))
    print("\n")

Imbalance ratio: 1:99
Imbalanced dataset metrics: Accuracy: 0.3667, Precision: 0.3667, Recall: 1.0000, F1-score: 0.5366
Balanced dataset with random over sampling metrics: Accuracy: 0.9667, Precision: 0.9167, Recall: 1.0000, F1-score: 0.9565
Balanced dataset with class weighting metrics: Accuracy: 0.9667, Precision: 0.9167, Recall: 1.0000, F1-score: 0.9565
Balanced dataset with xgb metrics: Accuracy: 0.8333, Precision: 0.6875, Recall: 1.0000, F1-score: 0.8148


Imbalance ratio: 5:95
Imbalanced dataset metrics: Accuracy: 0.7667, Precision: 0.6111, Recall: 1.0000, F1-score: 0.7586
Balanced dataset with random over sampling metrics: Accuracy: 0.8667, Precision: 0.7333, Recall: 1.0000, F1-score: 0.8462
Balanced dataset with class weighting metrics: Accuracy: 0.9000, Precision: 0.7857, Recall: 1.0000, F1-score: 0.8800
Balanced dataset with xgb metrics: Accuracy: 0.8667, Precision: 0.7333, Recall: 1.0000, F1-score: 0.8462


Imbalance ratio: 10:90
Imbalanced dataset metrics: Accuracy: 0.8000,